# Byte Pair Encoding (BPE) Tokenizer From Scratch (Simple)

This is a naive implementation for educational purposes. The [bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) notebook contains a more sophisticated implementation that matches the behavior in tiktoken.

## 1. The main idea behind byte pair encoding (BPE)

- LLMs cannot work directly with categorical data like text - you cannot perform mathematical operations on text
- The main idea in BPE is to convert text into an integer representation (token IDs) for LLM training

### 1.1 Bits and Bytes

- Before we dive into BPE algorithm, let's introduce the notion of bytes (BPE stands for **byte** pair encoding)
- Consider converting text into a byte array

In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


- When we call `list()` on a `bytearray` object, each byte is treated as an individual element, and the result is a list of integers corresponding to the byte values:

In [2]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- First intuition would be to assume that this is a valid way to convert text into a token ID representation that we need for embedding layer of an LLM
- However, the downside of this approach is that it is creating one ID for each character - this is a lot of IDs for a short text like this
- I.e., this means that for a 17-character input text, we have to use 17 token IDs as input to the LLM:

In [3]:
print(f"Number of characters: {len(text)}")
print(f"Number of token IDs: {len(ids)}")

Number of characters: 17
Number of token IDs: 17


- We know that the BPE tokenizers have a vocabulary where a token ID can represent whole words or subwords instead of each character
- For example, the GPT-2 tokenizer tokenizes the same text ("This is some text") into 4 instead of 17 tokens: `1212, 318, 617, 2420`
- You can double check this using the interactive [tiktoken app](https://platform.openai.com/tokenizer) or using the tiktoken library

In [4]:
import tiktoken

In [5]:
gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")

[1212, 318, 617, 2420]

- Since a byte consists of 8 bits, there are $2^8=256$ possible values that a single byte can represent, ranging from 0 to 255
- You can confirm this by executing the code `bytearray(range(0, 257))`, which will warn you that `ValueError: byte must be in range(0, 256)`

In [6]:
bytearray(range(0, 257))

ValueError: byte must be in range(0, 256)

- A BPE tokenizer usually uses these 256 values as its first 256 single-character tokens. We can visually check this by running the following code:

In [ ]:
for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i:<8} | {decoded:^8} | {len(decoded):>8}")
"""
0    |  !   |    1
1    |  "   |    1
2    |  #   |    1
...
255  |  �   |    1  # <---- single character tokens up to here
256  |   t  |    2
257  |   a  |    2
...
298  | ent  |    3
299  |   n  |    2
"""

- **Note:** Entries 256 and 257 are not single-character values but double-character values (a white space + a letter), which is a little shortcoming of the original GPT-2 BPE Tokenizer (this has been improved in the GPT-4 tokenizer)

### 1.2 Building the vocabulary

- The goal of BPE tokenization algorithm is to build a vocabulary of commonly occuring subwords like `298: ent` (which can be found in **entangle, entertain, enter, entrance, entity, ..., for example), or even complete words like:

In [8]:
for i in [318, 617, 1212, 2420]:
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i:<8} | {decoded:<8}")

318      |  is     
617      |  some   
1212     | This    
2420     |  text   


- The BPE algorithm was originally described in 1994: ["A New Algorithm for Data Compression"](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf) by Philip Gage
- Before we get to the actual code implementation, the form that is used for LLM tokenizers can be summarized as described in the following sections

### 1.3 BPE algorithm outline

**1. Identify frequent pairs**
- In each iteration, scan the text to find the most commonly occuring pair of bytes (or characters)

**2. Replace and record**
- Replace that pair with a new placeholder ID (one not already in use, e.g., if we start with 0...255, the first placeholder would be 256)
- Record this mapping in a lookup table
- The size of the lookup table is a hyperparameter, also called "vocabulary size" (for GPT-2, that's 50,257)

**3. Repeat until no gains**
- Keep repeating steps 1 and 2, continually merging the most frequent pairs
- Stop when no further compression is possible (e.g., no pair occurs more than once)

**Decompression (decoding)**
- To restore the original text, reverse the process by substituting each ID with its corresponding pair, using the lookup table

### 1.4 BPE algorithm example

#### 1.4.1 Concrete example of the encoding part (first three steps in section 1.3)
- Suppose we have the text (training dataset) `the cat in the hat` from which we want to build the vocabulary from a BPE tokenizer

**Iteration 1**
1. Identify frequent pairs
- In this text, "th" appears twice (at the beginning and before the second "e").
2. Replace and record
- Replace "th" with a new token ID that is not already in use, e.g. `256`.
- The new text is: `<256>e cat in <256>e hat`
- The new vocabulary is
```text
    0: ...
    ...
    256: "th"
```
**Iteration 2**
1. Identify frequent pairs
- In the text `<256>e cat in <256>e hat`, the pair `<256>e` appears twice.
2. Replace and record
- Replace `<256>e` with a new token ID that is not already in use, for example, `257`.
- The new text is: `<257> cat in <257> hat`
- The updated vocabulary is:
```text
    0: ...
    ...
    256: "th"
    257: "<256>e"
```
**Iteration 3**
1. Identify frequent pairs
- In the text `<257> cat in <257> hat`, the pair `<257> ` appears twice.
2. Replace and record
- Replace `<257> ` with a new token ID that is not already in use, for example, `258`.
- The new text is: `<258>cat in <258>hat`
- The updated vocabulary is:
```text
    0: ...
    ...
    256: "th"
    257: "<256>e"
    258: "<257> "
```
- Repeat until no gains or we reach vocabulary size

#### 1.4.2 Concrete example of the decoding part (last step in section 1.3)
- To restore the original text, we reverse the process by substituting each token ID with its corresponding pair in the reverse order they were introduced
- Start with the final compressed text: `<258>cat in <258>hat`
- Substitute `<258>` → `<257> `: `<257> cat in <257> hat`
- Substitute `<257>` → `<256>e`: `<256>e cat in <256>e hat`
- Substitute `<256>` → "th": `the cat in the hat`

## 2. A simple BPE implementation
- Below is an implementation of this algorithm described above as a Python class that mimics the `tiktoken` Python user interface
- Note that the encoding part above describes the original training step via `train()`; however, the `encode()` method works similarly (although it looks a bit more complicated because of the special token handling):

1. Split the input text into individual bytes
2. Repeatedly find & replace (merge) adjacent tokens (pairs) when they match any pair in the learned BPE merges (from highest to lowest "rank," i.e., in the order they were learned)
3. Continue merging until no more merges can be applied
4. The final list of token IDs is the encoded output

In [9]:
from typing import Literal
from collections import Counter, deque
from functools import lru_cache

class BPETokenizerSimple:
    def __init__(self) -> None:
        # Maps token_id to token_str
        self.vocab = {}
        # Maps token_str to token_id
        self.inverse_vocab = {}
        # Dictionary of BPE merges: {(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}
    
    def train(self, text:str, vocab_size:int, allowed_special:set={"<|endoftext|>"}) -> None:
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # Preprocess: Replace spaces with 'Ġ'
        # Note that Ġ is particularity of the GPT-2 BPE implementation
        # E.g., "Hello world" might be tokenized as ["Hello", "Ġworld"]
        # (GPT-4 BPE would tokenize it as ["Hello", " world"])
        processed_text = []
        for i, char in enumerate(text):
            if char == " " and i != 0:
                processed_text.append("Ġ")
            if char != " ":
                processed_text.append(char)
        processed_text = "".join(processed_text)

        # Initialize vocab with unique characters, including 'Ġ' if present
        # Start with the first 256 ASCII characters
        unique_chars = [chr(i) for i in range(256)]

        # Extend unique_chars with characters from processed_text that are not already included
        unique_chars.extend([char for char in sorted(set(processed_text)) if char not in unique_chars])

        # Optionally, ensure 'Ġ' is included if it is relevant to your text processing
        if 'Ġ' not in unique_chars:
            unique_chars.append('Ġ')
        
        # Create vocab and inverse vocab
        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # Add allowed special tokens
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id
        
        # Tokenize the processed_text into token IDs
        token_ids = [self.inverse_vocab[char] for char in processed_text]

        # BPE steps 1-3: Repeatedly find and replace frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            if len(token_ids) < 2:
                break
            pair_id = self.find_freq_pair(token_ids, mode="most")
            if pair_id is None: # No more pairs to merge. Stopping training.
                break

            updated = self.replace_pair(token_ids, pair_id, new_id)
            if updated == token_ids:
                break

            token_ids = updated
            self.bpe_merges[pair_id] = new_id
        
        # Build the vocabulary with merged tokens
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def encode(self, text: str) -> list[int]:
        """
        Encode the input text into a list of token IDs.

        Args:
            text (str): The text to encode.

        Returns:
            List[int]: The list of token IDs.
        """
        tokens = []
        # Split text into tokens, keeping newlines intact
        words = text.replace("\n", " \n ").split() # Ensure '\n' is treated as a separate token

        for i, word in enumerate(words):
            if i > 0 and not word.startswith("\n"):
                tokens.append("Ġ" + word) # Add 'Ġ' to words that follow a space or newline
            else:
                tokens.append(word) # Handle first word or standalone '\n'
        
        token_ids = []
        for token in tokens:
            if token in self.inverse_vocab:
                # token is contained in the vocabulary as is
                token_id = self.inverse_vocab[token]
                token_ids.append(token_id)
            else:
                # Attempt to handle subword tokenization via BPE
                sub_token_ids = self.tokenize_with_bpe(token)
                token_ids.extend(sub_token_ids)
        
        return token_ids
    
    def tokenize_with_bpe(self, token: str) -> list[int]:
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # Tokenize the token into individual characters (as initial token IDs)
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")
        
        can_merge = True
        while can_merge and len(token_ids) > 1:
            can_merge = False
            new_tokens = []
            i = 0
            while i < len(token_ids) - 1:
                pair = (token_ids[i], token_ids[i + 1])
                if pair in self.bpe_merges:
                    merged_token_id = self.bpe_merges[pair]
                    new_tokens.append(merged_token_id)
                    # Uncomment for educational purposes:
                    # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]})")
                    i += 2 # Skip the next token as it's merged
                    can_merge = True
                else:
                    new_tokens.append(token_ids[i])
                    i += 1
            if i < len(token_ids):
                new_tokens.append(token_ids[i])
            token_ids = new_tokens

        return token_ids
    
    def decode(self, token_ids: list[int]) -> str:
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        decoded_string = ""
        for token_id in token_ids:
            if token_id not in self.vocab:
                raise ValueError(f"Token ID {token_id} not found in vocab.")
            token = self.vocab[token_id]
            if token.startswith("Ġ"):
                # Replace 'Ġ' with space
                decoded_string += " " + token[1:]
            else:
                decoded_string += token
        return decoded_string
    
    @staticmethod
    def find_freq_pair(token_ids: list[int], mode: Literal["most", "least"] = "most") -> tuple[int, int]:
        if len(token_ids) < 2:
            return None
        
        pairs = Counter(zip(token_ids, token_ids[1:]))
        if not pairs:
            return None
        
        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")
    
    @staticmethod
    def replace_pair(token_ids: list[int], pair_id: tuple[int, int], new_id: int) -> list[int]:
        dq = deque(token_ids)
        replaced = []

        while dq:
            current = dq.popleft()
            if dq and (current, dq[0]) == pair_id:
                replaced.append(new_id)
                # Remove the 2nd token of the pair, 1st was already removed
                dq.popleft()
            else:
                replaced.append(current)
        
        return replaced

### Edge-case handling for short token sequences

BPE merges require adjacent token pairs.
If the token sequence has fewer than 2 items, no pair exists, so `find_freq_pair` returns `None` and training stops gracefully.

In [10]:
tok = BPETokenizerSimple()

assert tok.find_freq_pair([]) is None
assert tok.find_freq_pair([42]) is None

tok.train("", vocab_size=270)
tok.train("H", vocab_size=270)
tok.train("He", vocab_size=270)

print("Edge-case checks passed.")

Edge-case checks passed.


## 3. BPE implementation walkthrough

- In practice, it is recommended to use tiktoken library as implementation above focuses on readability and educational purposes, not on performance
- The usage is similar to tiktoken, except that tiktoken does not have training method
- Let's see how `BPETokenizerSimple` works by going through some examples below

### 3.1 Training, encoding and decoding

- Let's first load some text as our training dataset:

In [11]:
from pathlib import Path

file_path = Path("../01_main-chapter-code/the-verdict.txt")
assert file_path.exists(), "File not found."

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

- Next, let's initialise and train the BPE tokenizer with vocabulary size of 1,000
- Note that the vocabulary is already 255 by default due to the byte values discussed earlier, so we are only "learning" 745 vocabulary entries
- For comparison, the GPT-2 vocabulary is 50,257 tokens, the GPT-4 vocabulary is 100,256 (`cl100k_base` in tiktoken), and GPT-4o uses 199,997 tokens (`o200k_base` in tiktoken); they have all much bigger training sets compared to our simple example text above

In [12]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

In [13]:
# print(tokenizer.vocab)
print(len(tokenizer.vocab))

1000


- This vocabulary is created merging 742 times (~`1000 - len(range(0, 256))`)

In [14]:
print(len(tokenizer.bpe_merges))

742


- This means that the first 256 entries are single-character tokens

- Next, let's use the created merges via the `encode` method to encode some text:

In [15]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [16]:
print("Number of characters:", len(input_text))
print("Number of token IDs:", len(token_ids))

Number of characters: 42
Number of token IDs: 20


- From the lengths above, we can see that a 42-character sentence was encoded into 20 token IDs, effectively cutting the input length roughly in half compared to a character-byte-based encoding

- **Note:** The vocabulary itself is used in the `decode()` method, which allows us to map the token IDs back into text:

In [17]:
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [18]:
print(tokenizer.decode(token_ids))

Jack embraced beauty through art and life.


- Iterating over each token ID can give us a better understanding of how the token IDs are decoded via vocabulary:

In [19]:
for token_id in token_ids:
    print(f"{token_id:<6} -> {tokenizer.decode([token_id]):<6}")

424    -> Jack  
256    ->       
654    -> em    
531    -> br    
302    -> ac    
311    -> ed    
256    ->       
296    -> be    
97     -> a     
465    -> ut    
121    -> y     
595    ->  through
841    ->  ar   
116    -> t     
287    ->  a    
466    -> nd    
256    ->       
326    -> li    
972    -> fe    
46     -> .     


- As we can see, most token IDs represent 2-character subwords; that's because the training data text is very short with not that many repetitive words, and because we used a relatively small vocabulary size

- As a summary, calling `decode(encode())` should be able to reproduce arbitrary input texts:

In [20]:
tokenizer.decode(tokenizer.encode("This is some text."))

'This is some text.'